# ABBA-BABA (D-statistic) analysis with ANGSD

**Course:** Population Genomics and Population Structure Analysis  
**Instructor:** Nikolay Oskolkov  
**Platform:** Google Colab

In this notebook we will:

1. Install Conda and ANGSD in Google Colab.
2. Mount Google Drive and define the locations of the 1000 Genomes and Neanderthal data.
3. Build a BAM file list containing the Neanderthal genome and 10 CEU + 10 YRI low-coverage 1000 Genomes samples.
4. Run the ANGSD **ABBA-BABA / D-statistic** analysis.
5. Run the ANGSD `jackKnife.R` script and inspect the resulting D-statistics, jackknife estimates, standard errors and Z-scores.
6. Visualize the ABBA and BABA counts in R, following the procedure shown in the course slides.
7. Repeat the analysis with the three high-coverage genomes **Yoruba.bam, French.bam and Neanderthal.bam**.
8. Compare the low-coverage and high-coverage analyses and discuss the interpretation of the D-statistic.

The analysis uses the chimpanzee ancestral reference `anc.fa` to polarize alleles.

## 1. Install Conda and ANGSD

Google Colab can use a Conda environment for ANGSD. The installation below follows the same Conda-based approach used in the population-genomics notebooks.

> **Important:** the Conda installation restarts the Colab runtime automatically. After the restart, continue with the next cells.

In [ ]:
# Install Conda in Google Colab
!pip install -q condacolab
import condacolab
condacolab.install()

## 2. Install ANGSD

ANGSD will perform the ABBA-BABA calculation. The `-doAbbababa 1` analysis counts ABBA and BABA patterns in genomic blocks, while `-doCounts 1` enables base counting after the selected filters. The ancestral FASTA is supplied with `-anc`.

For human data, ANGSD uses a default block size of 5 Mb for this analysis; the block structure is then used by the jackknife procedure.

In [ ]:
# Install ANGSD and samtools from Bioconda
!conda install -y bioconda::angsd bioconda::samtools

# Check the installations
!angsd --version 2>&1 | head -n 3
!samtools --version | head -n 2


## 3. Install R and R magic

We use R inside the Python-based Colab notebook through **rpy2 R magic**. R is installed through Conda together with the packages needed for the visualizations.

This cell also installs `rpy2`, which provides the `%%R` notebook magic.

In [ ]:
# Install R for use with rpy2
!conda install -y -c conda-forge r-base

# Install the Python interface used by %%R
!pip install -q rpy2


In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R
cat("R version:", R.version.string, "\n")

## 4. Mount Google Drive and define paths

The low-coverage 1000 Genomes BAM files are located in:

`/content/drive/MyDrive/PopGen_PopStructure/1000G/smallbams`

The Neanderthal genome, the high-coverage Yoruba and French genomes, and `jackKnife.R` are located in:

`/content/drive/MyDrive/PopGen_PopStructure/Neanderthal`

The ancestral reference is expected at:

`/content/drive/MyDrive/PopGen_PopStructure/1000G/anc.fa`

All ABBA-BABA output files generated in this notebook are written to the `Neanderthal` directory.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os
import subprocess

PROJECT_DIR = Path("/content/drive/MyDrive/PopGen_PopStructure")
LOWCOV_DIR  = PROJECT_DIR / "1000G" / "smallbams"
NEAND_DIR   = PROJECT_DIR / "Neanderthal"
ANC_FA      = PROJECT_DIR / "1000G" / "anc.fa"

NEAND_BAM   = NEAND_DIR / "Neanderthal.bam"
YORUBA_BAM  = NEAND_DIR / "Yoruba.bam"
FRENCH_BAM  = NEAND_DIR / "French.bam"
JACKKNIFE_R = NEAND_DIR / "jackKnife.R"

print("Low-coverage BAM directory:", LOWCOV_DIR)
print("Neanderthal directory:     ", NEAND_DIR)
print("Ancestral reference:        ", ANC_FA)

# Check the important files/directories
for p in [LOWCOV_DIR, NEAND_DIR, ANC_FA, NEAND_BAM, YORUBA_BAM, FRENCH_BAM, JACKKNIFE_R]:
    print(f"{'OK  ' if p.exists() else 'MISS'} {p}")

### 4.1 Check the ancestral FASTA index

ANGSD needs the FASTA reference to be indexed. The index for `anc.fa` is normally:

`anc.fa.fai`

If it is missing, the next cell creates it with `samtools faidx`. This is the correct command for indexing a FASTA file; `samtools index` is for BAM/CRAM files.

In [ ]:
ANC_FAI = Path(str(ANC_FA) + ".fai")

if not ANC_FA.exists():
    raise FileNotFoundError(f"Missing ancestral FASTA: {ANC_FA}")

if not ANC_FAI.exists():
    print("FASTA index not found; creating:", ANC_FAI)
    subprocess.run(["samtools", "faidx", str(ANC_FA)], check=True)
else:
    print("FASTA index found:", ANC_FAI)

## 5. Prepare the low-coverage 1000 Genomes BAM list

The first analysis follows the procedure shown in the course slide.

The BAM list contains:

- the high-quality Neanderthal genome first;
- 10 YRI low-coverage BAM files;
- 10 CEU low-coverage BAM files.

The full Google Drive paths are written into the `bams` file. Using full paths means that ANGSD can be run from any working directory.

The resulting file is:

`/content/drive/MyDrive/PopGen_PopStructure/Neanderthal/bams`

In [ ]:
# Find and sort CEU and YRI BAM files
ceu_bams = sorted(LOWCOV_DIR.glob("*CEU*.bam"))
yri_bams = sorted(LOWCOV_DIR.glob("*YRI*.bam"))

print("CEU BAMs found:", len(ceu_bams))
print("YRI BAMs found:", len(yri_bams))

if len(ceu_bams) < 10:
    raise FileNotFoundError("Fewer than 10 CEU BAM files were found.")
if len(yri_bams) < 10:
    raise FileNotFoundError("Fewer than 10 YRI BAM files were found.")
if not NEAND_BAM.exists():
    raise FileNotFoundError(f"Missing Neanderthal BAM: {NEAND_BAM}")

LOWCOV_BAMS = NEAND_DIR / "bams"

selected_yri = yri_bams[:10]
selected_ceu = ceu_bams[:10]

with open(LOWCOV_BAMS, "w") as f:
    f.write(str(NEAND_BAM) + "\n")
    for bam in selected_yri:
        f.write(str(bam) + "\n")
    for bam in selected_ceu:
        f.write(str(bam) + "\n")

print(f"Created: {LOWCOV_BAMS}")
print(f"Number of BAMs: {sum(1 for _ in open(LOWCOV_BAMS))}")
print("\nFirst entries:")
print("\n".join(LOWCOV_BAMS.read_text().splitlines()[:5]))
print("\nLast entries:")
print("\n".join(LOWCOV_BAMS.read_text().splitlines()[-5:]))

### 5.1 Check BAM indexes

ANGSD needs indexed BAM files. We check for the corresponding `.bai` files and report any missing indexes.

In [ ]:
# Check indexes for all BAMs in the low-coverage analysis
lowcov_paths = [Path(x.strip()) for x in LOWCOV_BAMS.read_text().splitlines() if x.strip()]

missing_bai = []
for bam in lowcov_paths:
    bai1 = Path(str(bam) + ".bai")
    bai2 = bam.with_suffix(".bai")
    if not (bai1.exists() or bai2.exists()):
        missing_bai.append(bam)

if missing_bai:
    print("Missing BAM indexes:")
    for bam in missing_bai:
        print("  ", bam)
    print("\nIndex these BAMs before running ANGSD.")
else:
    print("All BAM files have BAM indexes.")

## 6. Run the low-coverage ABBA-BABA analysis

This is the ANGSD command corresponding to the first course slide:

```bash
angsd -out out -doAbbababa 1 -bam bams -doCounts 1 -anc anc.fa
```

Here we use absolute paths and write the output into the `Neanderthal` directory.

The analysis produces the `.abbababa` block-level output. Each block contains ABBA/BABA information for the possible three-individual combinations. The ancestral FASTA is used as the outgroup.

For the teaching example, we subsequently focus on the topology:

**(YRI, CEU, Neanderthal; chimpanzee)**

where the D-statistic tests asymmetry between ABBA and BABA patterns.

In [ ]:
# Run ANGSD ABBA-BABA analysis for the low-coverage 1000G data
LOWCOV_OUT = NEAND_DIR / "out_abbababa"

cmd = [
    "angsd",
    "-out", str(LOWCOV_OUT),
    "-doAbbababa", "1",
    "-bam", str(LOWCOV_BAMS),
    "-doCounts", "1",
    "-anc", str(ANC_FA)
]

print("Running:")
print(" ".join(cmd))

import subprocess
subprocess.run(cmd, check=True)

print("\nANGSD analysis completed.")
print("Output directory:", NEAND_DIR)

## 7. Run the block jackknife

ANGSD's `jackKnife.R` script calculates the D-statistic and its block-jackknife standard error and Z-score from the `.abbababa` output.

The script is deliberately taken from the Google Drive location specified for this course:

`/content/drive/MyDrive/PopGen_PopStructure/Neanderthal/jackKnife.R`

The output is written to the same `Neanderthal` directory.

In [ ]:
# Verify the jackknife script
if not JACKKNIFE_R.exists():
    raise FileNotFoundError(f"jackKnife.R not found: {JACKKNIFE_R}")

LOWCOV_JK_OUT = NEAND_DIR / "out_abbababa_results"

cmd = [
    "Rscript",
    str(JACKKNIFE_R),
    f"file={LOWCOV_OUT}.abbababa",
    f"indNames={LOWCOV_BAMS}",
    f"outfile={LOWCOV_JK_OUT}"
]

print("Running:")
print(" ".join(cmd))

subprocess.run(cmd, check=True)

print("\nJackknife completed.")

### 7.1 Inspect the jackknife output

The ANGSD jackknife output contains columns such as:

`H1`, `H2`, `H3`, `nABBA`, `nBABA`, `Dstat`, `jackEst`, `SE`, and `Z`.

The first rows are displayed below.

In [ ]:
# Find the jackknife output file
possible_results = [
    NEAND_DIR / "out_abbababa_results.txt",
    NEAND_DIR / "out_abbababa_results"
]

LOWCOV_RESULTS = next((p for p in possible_results if p.exists()), None)

if LOWCOV_RESULTS is None:
    raise FileNotFoundError(
        "Could not find the jackknife output. Expected "
        "out_abbababa_results.txt or out_abbababa_results."
    )

print("Jackknife result:", LOWCOV_RESULTS)
print(LOWCOV_RESULTS.read_text()[:5000])

## 8. Select the YRI–CEU–Neanderthal topology

Following the first slide, we select rows corresponding to:

- **H1 = YRI**
- **H2 = CEU**
- **H3 = Neanderthal**

The exact number of rows depends on the individuals included in the BAM list. The table below is the subset used for the visualization.

In [ ]:
import pandas as pd

df_low = pd.read_csv(LOWCOV_RESULTS, sep="\t")

print("Columns:")
print(df_low.columns.tolist())

# ANGSD/jackKnife output uses sample names in H1-H3.
# We search flexibly for Neanderthal/NEAND because the exact label can vary.
mask = (
    df_low["H1"].astype(str).str.contains("YRI", case=False, na=False)
    & df_low["H2"].astype(str).str.contains("CEU", case=False, na=False)
    & df_low["H3"].astype(str).str.contains("NEAND|NEANDER", case=False, regex=True, na=False)
)

df_low_topology = df_low.loc[mask].copy()

print(f"Selected rows: {len(df_low_topology)}")
if df_low_topology.empty:
    raise ValueError(
        "No YRI–CEU–Neanderthal rows were found. Inspect the H1/H2/H3 "
        "columns above and adjust the filter if the sample naming differs."
    )

display(df_low_topology)


## 9. Visualization in R: ABBA and BABA counts

We now reproduce the main visualization from the first slide: a boxplot comparing the numbers of ABBA and BABA sites for the selected YRI–CEU–Neanderthal comparisons.

A Wilcoxon rank-sum test is also shown, as in the teaching example.

The plot is generated directly in R using `%%R`.

In [ ]:
# Pass the selected table and output directory to R
NEAND_DIR_R = str(NEAND_DIR)
%R -i df_low_topology -i NEAND_DIR_R


In [ ]:
%%R -w 1400 -h 800 -u px

# Make sure the expected columns are numeric
df_low_topology$nABBA <- as.numeric(df_low_topology$nABBA)
df_low_topology$nBABA <- as.numeric(df_low_topology$nBABA)

png(
  filename = file.path(as.character(NEAND_DIR), "lowcov_ABBA_BABA_boxplot.png"),
  width = 1400, height = 800, res = 120
)

par(mar = c(5, 5, 4, 2))

boxplot(
  df_low_topology$nABBA,
  df_low_topology$nBABA,
  names = c("ABBA", "BABA"),
  ylab = "COUNTS",
  main = "ABBA and BABA counts: YRI–CEU–Neanderthal",
  col = 2
)

dev.off()

# Re-open a plotting device for inline display in Colab
par(mar = c(5, 5, 4, 2))
boxplot(
  df_low_topology$nABBA,
  df_low_topology$nBABA,
  names = c("ABBA", "BABA"),
  ylab = "COUNTS",
  main = "ABBA and BABA counts: YRI–CEU–Neanderthal",
  col = 2
)

cat("\nWilcoxon rank-sum test:\n")
print(wilcox.test(
  df_low_topology$nABBA,
  df_low_topology$nBABA
))

cat("\nSaved:", file.path(as.character(NEAND_DIR), "lowcov_ABBA_BABA_boxplot.png"), "\n")


### 9.1 Display the D-statistic results

For each selected comparison, we display the D-statistic, jackknife estimate, standard error and Z-score. A positive D-statistic means that ABBA counts exceed BABA counts for the specified topology; statistical support is assessed using the jackknife standard error and Z-score rather than the sign alone.

In [ ]:
%%R -w 1400 -h 800 -u px

keep <- c("H1", "H2", "H3", "nABBA", "nBABA",
          "Dstat", "jackEst", "SE", "Z")

print(df_low_topology[, intersect(keep, colnames(df_low_topology))],
      row.names = FALSE)

# Plot D-statistics with jackknife SE
d <- as.numeric(df_low_topology$Dstat)
se <- as.numeric(df_low_topology$SE)

png(
  filename = file.path(as.character(NEAND_DIR_R), "lowcov_Dstat_SE.png"),
  width = 1400, height = 800, res = 120
)

par(mar = c(8, 5, 4, 2))
plot(
  seq_along(d), d,
  pch = 19,
  xaxt = "n",
  ylim = range(c(d - se, d + se, 0), na.rm = TRUE),
  xlab = "",
  ylab = "D-statistic",
  main = "D-statistic with jackknife standard error"
)
segments(seq_along(d), d - se, seq_along(d), d + se)
abline(h = 0, lty = 2)

axis(
  1,
  at = seq_along(d),
  labels = paste(df_low_topology$H1,
                 df_low_topology$H2,
                 sep = " / "),
  las = 2
)

dev.off()

# Inline display
par(mar = c(8, 5, 4, 2))
plot(
  seq_along(d), d,
  pch = 19,
  xaxt = "n",
  ylim = range(c(d - se, d + se, 0), na.rm = TRUE),
  xlab = "",
  ylab = "D-statistic",
  main = "D-statistic with jackknife standard error"
)
segments(seq_along(d), d - se, seq_along(d), d + se)
abline(h = 0, lty = 2)
axis(
  1,
  at = seq_along(d),
  labels = paste(df_low_topology$H1,
                 df_low_topology$H2,
                 sep = " / "),
  las = 2
)

cat("\nSaved:", file.path(as.character(NEAND_DIR_R), "lowcov_Dstat_SE.png"), "\n")


## 10. High-coverage genomes: Yoruba, French and Neanderthal

The low-coverage 1000 Genomes data contain relatively few usable observations per site compared with high-coverage whole genomes. We therefore repeat the ABBA-BABA analysis with the three high-coverage genomes used in the second course slide:

- `Yoruba.bam`
- `French.bam`
- `Neanderthal.bam`

All three BAM files are located in:

`/content/drive/MyDrive/PopGen_PopStructure/Neanderthal`

We create a three-line BAM list in the order:

```text
Yoruba.bam
French.bam
Neanderthal.bam
```

With these three genomes, ANGSD evaluates the possible three-individual configurations. The first configuration corresponds to:

**(Yoruba, French, Neanderthal; chimpanzee)**

which is the topology used to illustrate Neanderthal gene flow into modern Europeans.

In [ ]:
HIGHCOV_BAMS = NEAND_DIR / "highcov_bams"

for p in [YORUBA_BAM, FRENCH_BAM, NEAND_BAM]:
    if not p.exists():
        raise FileNotFoundError(f"Missing high-coverage BAM: {p}")

with open(HIGHCOV_BAMS, "w") as f:
    f.write(str(YORUBA_BAM) + "\n")
    f.write(str(FRENCH_BAM) + "\n")
    f.write(str(NEAND_BAM) + "\n")

print(HIGHCOV_BAMS)
print(HIGHCOV_BAMS.read_text())

### 10.1 Run ANGSD for the three high-coverage genomes

This is the three-genome version of the command shown in the first slide:

```bash
angsd -out highcov_abbababa -doAbbababa 1 -bam highcov_bams -doCounts 1 -anc anc.fa
```

The output is again written to the `Neanderthal` directory.

In [ ]:
HIGHCOV_OUT = NEAND_DIR / "highcov_abbababa"

cmd = [
    "angsd",
    "-out", str(HIGHCOV_OUT),
    "-doAbbababa", "1",
    "-bam", str(HIGHCOV_BAMS),
    "-doCounts", "1",
    "-anc", str(ANC_FA)
]

print("Running:")
print(" ".join(cmd))

subprocess.run(cmd, check=True)

print("\nHigh-coverage ANGSD analysis completed.")

### 10.2 Jackknife for the high-coverage genomes

We use the same `jackKnife.R` script, but now with the three-genome BAM list.

In [ ]:
HIGHCOV_JK_OUT = NEAND_DIR / "highcov_abbababa_results"

cmd = [
    "Rscript",
    str(JACKKNIFE_R),
    f"file={HIGHCOV_OUT}.abbababa",
    f"indNames={HIGHCOV_BAMS}",
    f"outfile={HIGHCOV_JK_OUT}"
]

print("Running:")
print(" ".join(cmd))

subprocess.run(cmd, check=True)

possible_high = [
    NEAND_DIR / "highcov_abbababa_results.txt",
    NEAND_DIR / "highcov_abbababa_results"
]
HIGHCOV_RESULTS = next((p for p in possible_high if p.exists()), None)

if HIGHCOV_RESULTS is None:
    raise FileNotFoundError("High-coverage jackknife output was not found.")

print("\nResult file:", HIGHCOV_RESULTS)
print(HIGHCOV_RESULTS.read_text()[:5000])

## 11. Visualize the high-coverage D-statistics in R

For the three high-coverage genomes there are only three possible three-individual configurations. The first line corresponds to:

**Yoruba – French – Neanderthal**

and is the topology relevant to testing whether Neanderthal shares more derived alleles with French than with Yoruba.

We plot all three D-statistics with their jackknife standard errors.

In [ ]:
df_high = pd.read_csv(HIGHCOV_RESULTS, sep="\t")
display(df_high)

In [ ]:
NEAND_DIR_R = str(NEAND_DIR)
%R -i df_high -i NEAND_DIR_R

In [ ]:
%%R -w 1400 -h 800 -u px

df_high$Dstat <- as.numeric(df_high$Dstat)
df_high$SE <- as.numeric(df_high$SE)

x <- seq_len(nrow(df_high))
y <- df_high$Dstat
se <- df_high$SE

labels <- paste(df_high$H1, df_high$H2, df_high$H3, sep = " / ")

# Save a high-resolution PNG
png(
  filename = file.path(as.character(NEAND_DIR_R), "highcov_Dstat_SE.png"),
  width = 1400, height = 800, res = 120
)

par(mar = c(9, 5, 4, 2))

plot(
  x, y,
  pch = 19,
  xaxt = "n",
  ylim = range(c(y - se, y + se, 0), na.rm = TRUE),
  xlab = "",
  ylab = "D-statistic",
  main = "D-statistics: high-coverage genomes"
)

segments(x, y - se, x, y + se)
abline(h = 0, lty = 2)
axis(1, at = x, labels = labels, las = 2)

dev.off()

# Inline display
par(mar = c(9, 5, 4, 2))

plot(
  x, y,
  pch = 19,
  xaxt = "n",
  ylim = range(c(y - se, y + se, 0), na.rm = TRUE),
  xlab = "",
  ylab = "D-statistic",
  main = "D-statistics: high-coverage genomes"
)

segments(x, y - se, x, y + se)
abline(h = 0, lty = 2)
axis(1, at = x, labels = labels, las = 2)

cat("\nSaved:", file.path(as.character(NEAND_DIR_R), "highcov_Dstat_SE.png"), "\n")


## 12. Interpreting the high-coverage result

For the topology:

**(Yoruba, French, Neanderthal; chimpanzee)**

a positive D-statistic indicates an excess of ABBA relative to BABA patterns under the allele-polarization convention used by the analysis.

The magnitude of the D-statistic and its uncertainty should be considered together. In the course example, the high number of genomic sites produces a much smaller jackknife standard error than in the low-coverage example, giving a much larger absolute Z-score.

A statistically supported positive D-statistic for the Yoruba–French–Neanderthal topology is consistent with gene flow between Neanderthals and the French/European lineage. The D-statistic itself does not identify the direction or exact timing of gene flow, and the test is sensitive to the assumed topology and outgroup.

## 13. Files generated by this notebook

All analysis outputs are stored in:

`/content/drive/MyDrive/PopGen_PopStructure/Neanderthal`

Important files include:

- `bams` — low-coverage Neanderthal + YRI + CEU BAM list
- `out_abbababa.abbababa` — low-coverage ANGSD ABBA-BABA block output
- `out_abbababa_results.txt` — low-coverage jackknife results
- `highcov_bams` — three-genome high-coverage BAM list
- `highcov_abbababa.abbababa` — high-coverage ANGSD ABBA-BABA block output
- `highcov_abbababa_results.txt` — high-coverage jackknife results
- `jackKnife.R` — the jackknife script supplied in the course data directory

## 14. Summary

In this notebook we:

- installed ANGSD in Google Colab;
- enabled R through rpy2 `%%R` magic;
- assembled full-path BAM lists from the Google Drive;
- ran ANGSD `-doAbbababa 1` on low-coverage 1000 Genomes samples;
- used the supplied `jackKnife.R` script to obtain D-statistics, standard errors and Z-scores;
- visualized ABBA/BABA counts and D-statistics in R;
- repeated the analysis with high-coverage Yoruba, French and Neanderthal genomes;
- compared the precision of the low- and high-coverage analyses.

### References

Korneliussen, T. S., Albrechtsen, A., & Nielsen, R. (2014). ANGSD: Analysis of Next Generation Sequencing Data. *BMC Bioinformatics*.

ANGSD documentation: ABBA-BABA / D-statistic analysis.